# Example 05: Reasoning Mode Comparison（思考模式对比）

no_think / low / high 三档对比：回答质量、思考 token 数量、耗时

**Demonstrates:**
- The three `reasoning_effort` modes: `no_think` / `low` / `high`
- How to read reasoning tokens (`delta.reasoning_content`) from the response
- Side-by-side comparison of answer quality and token cost

**Prerequisites:**
```bash
pip install openai
# Start the Hy3 server with --reasoning-parser hy_v3 (vLLM) or --reasoning-parser hunyuan (SGLang)
```

In [ ]:
import os
import time
from openai import OpenAI

client = OpenAI(
    base_url=os.environ.get("HY3_BASE_URL", "http://127.0.0.1:8000/v1"),
    api_key=os.environ.get("HY3_API_KEY", "EMPTY"),
    timeout=300,
)

MODEL = os.environ.get("HY3_MODEL", "hy3")


def ask(prompt: str, reasoning_effort: str) -> dict:
    """Send a prompt with the given reasoning_effort; return timing + content."""
    t_start = time.perf_counter()
    thinking_content = ""
    answer_content = ""
    completion_tokens = 0

    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.9,
        top_p=1.0,
        stream=True,
        stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"reasoning_effort": reasoning_effort}},
    )

    for chunk in stream:
        if chunk.usage is not None:
            completion_tokens = chunk.usage.completion_tokens
            continue
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        reasoning_delta = getattr(delta, "reasoning_content", None)
        if reasoning_delta:
            thinking_content += reasoning_delta
        if delta.content:
            answer_content += delta.content

    return {
        "reasoning_effort": reasoning_effort,
        "thinking": thinking_content,
        "answer": answer_content,
        "completion_tokens": completion_tokens,
        "elapsed_s": time.perf_counter() - t_start,
    }


def print_result(result: dict):
    print(f"\n{'─' * 60}")
    print(f"  reasoning_effort = '{result['reasoning_effort']}'")
    print(f"{'─' * 60}")
    if result["thinking"]:
        preview = result["thinking"][:300]
        if len(result["thinking"]) > 300:
            preview += f"... [{len(result['thinking'])} chars total]"
        print(f"[Thinking]\n{preview}\n")
    else:
        print("[Thinking] (none — direct response)\n")
    print(f"[Answer]\n{result['answer']}")
    print(f"\nCompletion tokens : {result['completion_tokens']}")
    print(f"Elapsed           : {result['elapsed_s']:.2f}s")

## Test 1: Simple Factual Question（简单事实问题）

简单问题三档差异不明显，`no_think` 最快。

In [ ]:
simple_prompt = "What is the boiling point of water at sea level?"
print(f"Prompt: {simple_prompt}")

for effort in ["no_think", "low", "high"]:
    print_result(ask(simple_prompt, effort))

**Sample output:**
```
────────────────────────────────────────────────────────────
  reasoning_effort = 'no_think'
────────────────────────────────────────────────────────────
[Thinking] (none — direct response)

[Answer]
Water boils at 100°C (212°F) at sea level.
Completion tokens : 14
Elapsed           : 1.2s
```

## Test 2: Math Word Problem（数学推理）

`high` 模式通过完整推理链显著降低出错率。

In [ ]:
math_prompt = (
    "A train leaves Station A at 08:00 traveling at 120 km/h toward Station B. "
    "Another train leaves Station B at 08:30 traveling at 90 km/h toward Station A. "
    "The stations are 420 km apart. At what time do the trains meet?"
)
print(f"Prompt: {math_prompt}")

for effort in ["no_think", "high"]:
    print_result(ask(math_prompt, effort))

**Sample output (high mode):**
```
────────────────────────────────────────────────────────────
  reasoning_effort = 'high'
────────────────────────────────────────────────────────────
[Thinking]
By 08:30 Train A has traveled: 0.5 h × 120 = 60 km.
Remaining gap: 420 − 60 = 360 km.
Closing speed: 120 + 90 = 210 km/h.
Time to close: 360 / 210 = 12/7 h ≈ 1 h 42 min 51 s.
Meet time: 08:30 + 1:42:51 ≈ 10:12:51 ... [847 chars total]

[Answer]
The trains meet at approximately **10:13 AM**.

Completion tokens : 312
Elapsed           : 18.4s
```

## Test 3: Code Generation（代码生成）

In [ ]:
code_prompt = (
    "Write a Python function that finds all prime numbers up to N "
    "using the Sieve of Eratosthenes. Include type hints and a brief docstring."
)
print(f"Prompt: {code_prompt}")

for effort in ["no_think", "high"]:
    print_result(ask(code_prompt, effort))

**Sample output:**
```
────────────────────────────────────────────────────────────
  reasoning_effort = 'no_think'
────────────────────────────────────────────────────────────
[Thinking] (none — direct response)

[Answer]
def sieve_of_eratosthenes(n: int) -> list[int]:
    """Return all prime numbers up to n (inclusive)."""
    if n < 2:
        return []
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False
    for i in range(2, int(n**0.5) + 1):
        if is_prime[i]:
            for j in range(i * i, n + 1, i):
                is_prime[j] = False
    return [i for i, p in enumerate(is_prime) if p]

Completion tokens : 98
Elapsed           : 3.1s

────────────────────────────────────────────────────────────
  reasoning_effort = 'high'
────────────────────────────────────────────────────────────
[Thinking]
The Sieve of Eratosthenes works by iteratively marking multiples of each
prime starting from 2. I'll initialize a boolean array, mark non-primes,
and collect indices that remain True ... [312 chars total]

[Answer]
def sieve_of_eratosthenes(n: int) -> list[int]:
    """
    Return a list of all prime numbers up to and including n.
    Uses the Sieve of Eratosthenes algorithm: O(n log log n) time.
    """
    if n < 2:
        return []
    is_prime = bytearray([1]) * (n + 1)
    is_prime[0] = is_prime[1] = 0
    for i in range(2, int(n**0.5) + 1):
        if is_prime[i]:
            is_prime[i*i::i] = bytearray(len(is_prime[i*i::i]))
    return [i for i, p in enumerate(is_prime) if p]

Completion tokens : 187
Elapsed           : 11.8s
```

## Summary（选用指南）

| Mode | 思考 token | 适用场景 |
|---|---|---|
| `no_think` | 0 | 聊天、翻译、摘要、简单问答 |
| `low` | 少量 | 轻量分析、短代码生成 |
| `high` | 大量 | 数学证明、复杂代码、多步推理、Agent 任务 |

> **建议**：从 `no_think` 开始，任务失败或需要逐步正确性保证时再升为 `high`。